<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W6D5_Mini_Project_Sentiment_Assistant_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Mini Project — Sentiment Assistant with BERT Fine-Tuning

**Developers Institute & Sira Labs — Week 6, Day 5**

## Contexte

Une équipe de support client souhaite détecter automatiquement le sentiment des
commentaires longs afin d’identifier rapidement les clients insatisfaits et de
réduire le risque de désabonnement.

## Objectifs

Dans ce notebook, nous allons :

1. charger le dataset IMDB ;
2. préparer les textes avec le tokenizer de BERT ;
3. fine-tuner `bert-base-uncased` pour une classification binaire ;
4. suivre les performances sur un jeu de validation ;
5. évaluer le modèle sur un jeu de test intact ;
6. créer une fonction réutilisable de prédiction ;
7. relier les résultats à un cas d’usage de support client.

> **Google Colab :** activez un GPU avec  
> `Exécution → Modifier le type d’exécution → T4 GPU`.


In [ ]:

# Installation de versions compatibles avec les modèles TensorFlow de Transformers
%pip install -q "tensorflow==2.18.0" "tf-keras==2.18.0" \
    "tensorflow-datasets==4.9.7" "transformers==4.48.3" \
    "accelerate==1.3.0" "evaluate==0.4.3"


In [ ]:

# IMPORTANT : cette variable doit être définie avant d'importer TensorFlow
# et Transformers afin d'utiliser l'API Keras compatible.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import platform
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
import transformers
import evaluate

from transformers import BertTokenizer, TFBertForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python version       :", platform.python_version())
print("TensorFlow version   :", tf.__version__)
print("Transformers version :", transformers.__version__)
print("GPU détecté          :", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print(
        "\nAttention : aucun GPU n'est détecté. "
        "L'entraînement complet sera beaucoup plus lent sur CPU."
    )



## 1. Chargement et séparation du dataset IMDB

Le dataset IMDB contient des critiques de films associées à deux classes :

- `0` : sentiment négatif ;
- `1` : sentiment positif.

Nous utilisons :

- 80 % du split d’entraînement pour apprendre ;
- 20 % du split d’entraînement pour valider ;
- le split de test complet uniquement pour l’évaluation finale.

Cette séparation évite d’utiliser le jeu de test pendant l’apprentissage.


In [ ]:

(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    "imdb_reviews",
    split=[
        "train[:80%]",
        "train[80%:]",
        "test",
    ],
    as_supervised=True,
    with_info=True,
)

print(ds_info)
print("\nTailles des jeux de données :")
print("Entraînement :", tf.data.experimental.cardinality(ds_train_raw).numpy())
print("Validation   :", tf.data.experimental.cardinality(ds_val_raw).numpy())
print("Test         :", tf.data.experimental.cardinality(ds_test_raw).numpy())

print("\nExemples :")
for text, label in ds_train_raw.take(2):
    sentiment = "Positive" if int(label.numpy()) == 1 else "Negative"
    review = text.numpy().decode("utf-8")
    print(f"Label : {sentiment}")
    print(review[:300], "...\n")



## 2. Tokenisation et création du pipeline TensorFlow

BERT utilise une tokenisation **WordPiece**. Le tokenizer :

- ajoute les tokens spéciaux `[CLS]` et `[SEP]` ;
- transforme le texte en identifiants numériques ;
- complète ou tronque chaque texte à une longueur fixe ;
- crée un masque d’attention pour distinguer les vrais tokens du remplissage.


In [ ]:

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME,
    do_lower_case=True,
)

print("Tokenizer chargé :", tokenizer.name_or_path)
print("Taille du vocabulaire :", tokenizer.vocab_size)


def encode_review_py(review_tensor):
    """Tokenise un texte reçu depuis tf.py_function."""
    review_bytes = review_tensor.numpy()
    review_text = review_bytes.decode("utf-8")

    encoded = tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

    return (
        np.asarray(encoded["input_ids"], dtype=np.int32),
        np.asarray(encoded["attention_mask"], dtype=np.int32),
        np.asarray(encoded["token_type_ids"], dtype=np.int32),
    )


def tf_encode(text, label):
    input_ids, attention_mask, token_type_ids = tf.py_function(
        func=encode_review_py,
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )

    # tf.py_function ne conserve pas automatiquement les dimensions.
    input_ids.set_shape([MAX_LENGTH])
    attention_mask.set_shape([MAX_LENGTH])
    token_type_ids.set_shape([MAX_LENGTH])

    features = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }

    return features, tf.cast(label, tf.int32)


def prepare_dataset(dataset, training=False):
    dataset = dataset.map(
        tf_encode,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=5000,
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    return (
        dataset
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = prepare_dataset(ds_train_raw, training=True)
val_ds = prepare_dataset(ds_val_raw)
test_ds = prepare_dataset(ds_test_raw)

sample_features, sample_labels = next(iter(train_ds))

print("\nDimensions d'un batch :")
print("input_ids      :", sample_features["input_ids"].shape)
print("attention_mask :", sample_features["attention_mask"].shape)
print("token_type_ids :", sample_features["token_type_ids"].shape)
print("labels         :", sample_labels.shape)



## 3. Initialisation du modèle BERT

`TFBertForSequenceClassification` contient :

1. l’encodeur BERT pré-entraîné ;
2. une tête de classification produisant deux logits.

Un faible taux d’apprentissage est utilisé afin de ne pas dégrader brutalement
les représentations déjà apprises par BERT.


In [ ]:

model = TFBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    use_safetensors=False,
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5,
    epsilon=1e-8,
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
)

metrics = [
    tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
]

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=metrics,
)

# Construire le modèle avant d'afficher son résumé
_ = model(
    {
        "input_ids": tf.zeros((1, MAX_LENGTH), dtype=tf.int32),
        "attention_mask": tf.ones((1, MAX_LENGTH), dtype=tf.int32),
        "token_type_ids": tf.zeros((1, MAX_LENGTH), dtype=tf.int32),
    }
)

model.summary()



## 4. Fine-tuning et suivi des performances

Nous entraînons BERT pendant deux époques. La validation permet de vérifier si
le modèle continue de progresser ou commence à surapprendre.


In [ ]:

EPOCHS = 2

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=1,
        restore_best_weights=True,
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:

# Courbe de perte
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], marker="o", label="Entraînement")
plt.plot(history.history["val_loss"], marker="o", label="Validation")
plt.title("Évolution de la perte")
plt.xlabel("Époque")
plt.ylabel("Perte")
plt.xticks(range(len(history.history["loss"])))
plt.legend()
plt.grid(True)
plt.show()

# Courbe d'accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], marker="o", label="Entraînement")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation")
plt.title("Évolution de l'accuracy")
plt.xlabel("Époque")
plt.ylabel("Accuracy")
plt.xticks(range(len(history.history["accuracy"])))
plt.legend()
plt.grid(True)
plt.show()



## 5. Évaluation sur le jeu de test

Le jeu de test n’a pas été utilisé pour ajuster les paramètres du modèle. Son
score donne donc une estimation plus honnête de la capacité de généralisation.


In [ ]:

eval_metrics = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)

print("\nRésultats sur le jeu de test :")
for metric_name, metric_value in eval_metrics.items():
    print(f"{metric_name:10s}: {metric_value:.4f}")

test_accuracy = eval_metrics["accuracy"]

if test_accuracy >= 0.90:
    print("\nLe modèle atteint le benchmark pédagogique de 90 %.")
else:
    print(
        "\nLe score est inférieur à 90 %. "
        "On peut tester une époque supplémentaire, nettoyer les données "
        "ou ajuster le taux d'apprentissage."
    )



## 6. Fonction d’inférence réutilisable

La fonction suivante transforme un texte libre, exécute le modèle et retourne :

- le sentiment prédit ;
- la confiance associée à la classe choisie.


In [ ]:

def predict_sentiment(text: str):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Le texte doit être une chaîne non vide.")

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf",
    )

    outputs = model(encoded, training=False)
    probabilities = tf.nn.softmax(outputs.logits, axis=-1)[0].numpy()

    predicted_index = int(np.argmax(probabilities))
    label = "Positive" if predicted_index == 1 else "Negative"
    confidence = float(probabilities[predicted_index])

    return label, confidence


custom_sentence = (
    "The onboarding emails were confusing, "
    "but the agent fixed everything politely."
)

label, confidence = predict_sentiment(custom_sentence)

print("Texte :", custom_sentence)
print(f"Prédiction : {label} (confiance={confidence:.3f})")


In [ ]:

# Démonstration avec des messages proches d'un contexte de support client
support_messages = [
    "Your support team solved my issue quickly and professionally.",
    "I have contacted you three times and my account is still blocked.",
    "The service works, but the instructions are difficult to understand.",
    "I am extremely disappointed and I want to cancel my subscription.",
]

for message in support_messages:
    label, confidence = predict_sentiment(message)
    decision = (
        "Escalade conseillée"
        if label == "Negative" and confidence >= 0.75
        else "Revue normale"
    )

    print(f"Message    : {message}")
    print(f"Sentiment  : {label}")
    print(f"Confiance  : {confidence:.3f}")
    print(f"Décision   : {decision}")
    print("-" * 70)



## 7. Réflexion et prochaines étapes

### Quel levier a le plus amélioré les résultats ?

Le levier principal est le **fine-tuning avec un faible taux d’apprentissage**.
BERT possède déjà des représentations linguistiques générales ; l’objectif est
donc de les adapter progressivement à la classification de sentiments sans les
détériorer. La qualité du découpage des données, la troncature à 256 tokens et
le suivi de la validation contribuent également à la stabilité du modèle.

Une époque supplémentaire peut améliorer les résultats si la perte de
validation continue de diminuer. En revanche, elle risque d’augmenter le
surapprentissage si la perte de validation commence à remonter.

### Où ajouter des garde-fous avant un déploiement réel ?

Avant le déploiement, il faudrait :

- définir un seuil minimal de confiance avant toute décision automatique ;
- envoyer les prédictions incertaines vers un agent humain ;
- vérifier régulièrement les faux négatifs, car un client réellement
  mécontent ne doit pas être ignoré ;
- protéger les données personnelles présentes dans les messages ;
- surveiller la dérive des données et les différences de performance selon le
  type de client, la langue et le canal de communication ;
- empêcher le modèle de déclencher seul une sanction ou une décision critique.

Le sentiment doit rester un **signal d’aide à la décision**, et non une décision
finale autonome.

### Quels acteurs bénéficient le plus de cette solution ?

Le **responsable du support** bénéficie directement du système, car il peut
prioriser les conversations urgentes et suivre l’évolution de la satisfaction.

Le **product manager** peut regrouper les retours négatifs afin d’identifier les
fonctionnalités ou étapes du parcours utilisateur qui créent le plus de
frustration.

Le **responsable conformité** veille à ce que les données soient traitées de
manière sécurisée, que les décisions restent explicables et que les utilisateurs
ne soient pas pénalisés uniquement sur la base d’une prédiction automatique.

## Conclusion

Ce projet montre comment transformer un modèle BERT généraliste en assistant de
classification spécialisé. Le modèle peut servir à détecter les retours
négatifs, aider les équipes de support à prioriser les demandes et fournir des
indicateurs utiles aux équipes produit. Pour une utilisation réelle, il doit
être accompagné d’un contrôle humain, de seuils de confiance et d’un suivi
continu de ses performances.



## Soumission

1. Exécuter toutes les cellules du notebook.
2. Vérifier que les résultats et les courbes sont visibles.
3. Enregistrer une copie dans Google Drive.
4. Partager le notebook avec l’option **Tous les utilisateurs disposant du lien**.
5. Copier le lien public sur la plateforme DI.
6. Facultatif : utiliser `Fichier → Enregistrer une copie dans GitHub`.
